In [ ]:
!pip -q install datasets

Literature Data

In [ ]:
from huggingface_hub import login

login(" ")

In [ ]:
from datasets import load_dataset

ds = load_dataset("Vacaspati/Vacaspati")

print(ds)

print("\nColumns:")
print(ds["train"].column_names)

print("\nSample:")
print(ds["train"][0])

In [ ]:
import unicodedata
import re

count = 0

with open("bn_literature.txt", "w", encoding="utf-8") as out:

    for row in ds["train"]:

        text = row["text"]

        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        out.write(text + "\n")
        count += 1

print("Saved documents:", count)

In [ ]:
import random

random.seed(42)

TARGET = 250000

with open("bn_literature.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_literature.txt", encoding="utf-8") as inp, \
     open("bn_literature_250k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:

            out.write(line)
            count += 1

print("Saved:", count)

In [ ]:
import os

print(
    "Size MB:",
    round(
        os.path.getsize(
            "bn_literature_250k.txt"
        )/(1024**2),
        2
    )
)

In [ ]:
vocab = set()

chars = 0
words = 0

with open("bn_literature_250k.txt", encoding="utf-8") as f:

    for line in f:

        chars += len(line)

        toks = line.split()

        words += len(toks)

        vocab.update(toks)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))

print("TTR :", len(vocab)/words)

print(
    "Average Word Length:",
    chars / words
)

News Data

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
print(dataset)
print(len(dataset["train"]))

In [ ]:
import time

start = time.time()

for i, row in enumerate(dataset["train"]):
    if i % 100000 == 0:
        print(i, "rows processed")

    if i == 500000:
        break

print("Time:", time.time() - start)

In [ ]:
import random

TOTAL = len(dataset["train"])

random.seed(42)

selected = set(
    random.sample(
        range(TOTAL),
        125000
    )
)

with open(
    "news_125k.txt",
    "w",
    encoding="utf-8"
) as out:

    count = 0

    for idx, row in enumerate(dataset["train"]):

        if idx in selected:

            text = row["text"].strip()

            if text:

                out.write(text + "\n")

                count += 1

print("Saved:", count)

In [ ]:
import os

print(
    "Size MB:",
    round(
        os.path.getsize(
            "news_125k.txt"
        )/(1024**2),
        2
    )
)

In [ ]:
count = 0

for row in dataset["train"]:

    txt = row["text"].strip()

    if txt:
        count += 1

print("Non-empty docs:", count)

In [ ]:
import random

TARGET = 125000
random.seed(42)

reservoir = []
seen = 0

for row in dataset["train"]:

    text = row["text"].strip()

    if not text:
        continue

    seen += 1

    if len(reservoir) < TARGET:

        reservoir.append(text)

    else:

        j = random.randint(0, seen - 1)

        if j < TARGET:
            reservoir[j] = text

print("Non-empty seen:", seen)
print("Sample size:", len(reservoir))

with open(
    "news_125k.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in reservoir:
        f.write(doc + "\n")

print("Saved news_125k.txt")

In [ ]:
import os

for f in sorted(os.listdir()):
    if f.endswith(".txt"):
        print(f)

In [ ]:
import random

random.seed(42)

with open(
    "bn_literature_250k.txt",
    encoding="utf-8"
) as f:

    docs = f.readlines()

print("Available:", len(docs))

selected = random.sample(
    docs,
    125000
)

with open(
    "literature_125k.txt",
    "w",
    encoding="utf-8"
) as out:

    out.writelines(selected)

print("Saved:", len(selected))

In [ ]:
with open(
    "mixed_250k.txt",
    "w",
    encoding="utf-8"
) as out:

    for fname in [
        "news_125k.txt",
        "literature_125k.txt"
    ]:

        with open(
            fname,
            encoding="utf-8"
        ) as f:

            for line in f:
                out.write(line)

print("Merged successfully")

In [ ]:
with open(
    "mixed_250k.txt",
    "w",
    encoding="utf-8"
) as out:

    for fname in [
        "news_125k.txt",
        "literature_125k.txt"
    ]:

        with open(
            fname,
            encoding="utf-8"
        ) as f:

            for line in f:
                out.write(line)

print("Merged")

In [ ]:
count = 0

with open(
    "mixed_250k.txt",
    encoding="utf-8"
) as f:

    for _ in f:
        count += 1

print("Documents:", count)

In [ ]:
from collections import Counter

chars = 0
words = 0

vocab = Counter()

with open(
    "mixed_250k.txt",
    encoding="utf-8"
) as f:

    for line in f:

        chars += len(line)

        tokens = line.split()

        words += len(tokens)

        vocab.update(tokens)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))
print("TTR :", len(vocab)/words)
print(
    "Average Word Length:",
    chars/words
)

In [ ]:
import random

random.seed(42)

with open(
    "mixed_250k.txt",
    encoding="utf-8"
) as f:

    docs = f.readlines()

random.shuffle(docs)

split = int(
    0.9 * len(docs)
)

train = docs[:split]
test = docs[split:]

with open(
    "train_mix_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    f.writelines(train)

with open(
    "test_mix_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    f.writelines(test)

print("Train:", len(train))
print("Test :", len(test))

In [ ]:
import os

print(
    "Train MB:",
    round(
        os.path.getsize(
            "train_mix_bn.txt"
        )/(1024**2),
        2
    )
)

print(
    "Test MB:",
    round(
        os.path.getsize(
            "test_mix_bn.txt"
        )/(1024**2),
        2
    )
)

Transliteration

In [ ]:
!pip -q install aksharamukha

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()

count = 0

with open(
    "train_mix_bn.txt",
    encoding="utf-8"
) as fin, open(
    "train_mix_iso.txt",
    "w",
    encoding="utf-8"
) as fout:

    for line in fin:

        text = line.strip()

        iso = transliterate.process(
            "Bengali",
            "ISO",
            text
        )

        fout.write(
            iso + "\n"
        )

        count += 1

        if count % 10000 == 0:

            elapsed = (
                time.time()
                - start
            )

            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print(
    "Completed:",
    count
)

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()

count = 0

with open(
    "test_mix_bn.txt",
    encoding="utf-8"
) as fin, open(
    "test_mix_iso.txt",
    "w",
    encoding="utf-8"
) as fout:

    for line in fin:

        text = line.strip()

        iso = transliterate.process(
            "Bengali",
            "ISO",
            text
        )

        fout.write(
            iso + "\n"
        )

        count += 1

        if count % 10000 == 0:

            elapsed = (
                time.time()
                - start
            )

            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print(
    "Completed:",
    count
)

In [ ]:
import os

for fname in [
    "train_mix_bn.txt",
    "train_mix_iso.txt",
    "test_mix_bn.txt",
    "test_mix_iso.txt"
]:

    print(
        fname,
        ":",
        round(
            os.path.getsize(fname)
            /(1024**2),
            2
        ),
        "MB"
    )

In [ ]:
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0

    vocab = Counter()

    with open(
        path,
        encoding="utf-8"
    ) as f:

        for line in f:

            chars += len(line)

            toks = line.split()

            words += len(toks)

            vocab.update(toks)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars/words
    }

bn = corpus_stats(
    "train_mix_bn.txt"
)

iso = corpus_stats(
    "train_mix_iso.txt"
)

print("Bengali:", bn)
print("ISO:", iso)

print(
    "\nExpansion Factor:",
    iso["chars"] /
    bn["chars"]
)

print(
    "Vocabulary Ratio:",
    iso["vocab"] /
    bn["vocab"]
)

In [ ]:
from aksharamukha import transliterate
import random

random.seed(42)

with open("test_mix_bn.txt", encoding="utf-8") as f:
    docs = [x.strip() for x in f if x.strip()]

samples = random.sample(docs, 20)

for i, text in enumerate(samples, 1):

    iso = transliterate.process(
        "Bengali",
        "ISO",
        text
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    print("="*80)
    print(f"Sample {i}")
    print("\nOriginal:")
    print(text[:300])

    print("\nISO:")
    print(iso[:300])

    print("\nBack:")
    print(back[:300])

In [ ]:
bn_chars = set()

with open(
    "train_mix_bn.txt",
    encoding="utf-8"
) as f:

    for line in f:
        bn_chars.update(line)

covered = set()

with open(
    "train_mix_iso.txt",
    encoding="utf-8"
) as f:

    for line in f:

        back = transliterate.process(
            "ISO",
            "Bengali",
            line
        )

        covered.update(back)

coverage = len(
    bn_chars & covered
) / len(bn_chars)

missing = sorted(
    bn_chars - covered
)

print("Letter Coverage:", coverage)
print("Missing:", missing[:50])

In [ ]:
import random

random.seed(42)

words = []

with open(
    "test_mix_bn.txt",
    encoding="utf-8"
) as f:

    for line in f:
        words.extend(line.split())

sample_words = random.sample(
    words,
    100
)

correct = 0

for word in sample_words:

    iso = transliterate.process(
        "Bengali",
        "ISO",
        word
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    if word == back:
        correct += 1

print(
    "Exact Match:",
    correct,
    "/",
    len(sample_words)
)

print(
    "Accuracy:",
    correct / len(sample_words)
)

In [ ]:
for word in sample_words:

    iso = transliterate.process(
        "Bengali",
        "ISO",
        word
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    if word != back:

        print("\nORIGINAL:", word)
        print("ISO     :", iso)
        print("BACK    :", back)

Tokenization

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_mix_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_mix_bn_{vocab_size}.json"
    )

print("Finished")

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(
    tokenizer_file,
    test_file,
    vocab_size
):

    tokenizer = Tokenizer.from_file(
        tokenizer_file
    )

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                tokens = tokenizer.encode(
                    word
                ).tokens

                total_tokens += len(tokens)

                sent_tokens += len(tokens)

                total_chars += len(word)

                if len(tokens) > 1:
                    fragmented += 1

                if "[UNK]" in tokens:
                    oov += 1

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_mix_bn_5000.json"
)

words = [
    "বাংলাদেশ",
    "কলকাতা",
    "সাহিত্য",
    "কবিতা",
    "উপন্যাস"
]

for w in words:

    print("\n", w)

    print(
        tok.encode(w).tokens
    )

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results = []

for vocab_size in VOCABS:

    res = evaluate_tokenizer(
        f"bpe_mix_bn_{vocab_size}.json",
        "test_mix_bn.txt",
        vocab_size
    )

    print(res)

    results.append(res)

print("\nFinished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_mix_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_mix_iso_{vocab_size}.json"
    )

print("Finished")

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_mix_iso_5000.json"
)

words = [
    "bāṃlā",
    "kalakātā",
    "sāhitya",
    "kabitā",
    "upanyāsa"
]

for w in words:

    print("\n", w)

    print(
        tok.encode(w).tokens
    )

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(
    tokenizer_file,
    test_file,
    vocab_size
):

    tokenizer = Tokenizer.from_file(
        tokenizer_file
    )

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                tokens = tokenizer.encode(
                    word
                ).tokens

                total_tokens += len(tokens)

                sent_tokens += len(tokens)

                total_chars += len(word)

                if len(tokens) > 1:
                    fragmented += 1

                if "[UNK]" in tokens:
                    oov += 1

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

iso_bpe_results = []

for vocab_size in VOCABS:

    res = evaluate_tokenizer(
        f"bpe_mix_iso_{vocab_size}.json",
        "test_mix_iso.txt",
        vocab_size
    )

    print(res)

    iso_bpe_results.append(res)

print("\nFinished.")

In [ ]:
import csv

with open("mix_bn_bpe_results.csv", "w", newline="") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "vocab",
            "oov",
            "fertility",
            "cpt",
            "compression",
            "avg_tokens",
            "wfr",
            "variance"
        ]
    )

    writer.writeheader()

    writer.writerows(results)

print("Saved")

In [ ]:
import csv

with open("mix_iso_bpe_results.csv", "w", newline="") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "vocab",
            "oov",
            "fertility",
            "cpt",
            "compression",
            "avg_tokens",
            "wfr",
            "variance"
        ]
    )

    writer.writeheader()

    writer.writerows(results)

print("Saved")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining Bengali WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_mix_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_mix_bn_{vocab_size}.json"
    )

print("Finished")

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "wp_mix_bn_5000.json"
)

words = [
    "বাংলাদেশ",
    "কলকাতা",
    "সাহিত্য",
    "কবিতা",
    "উপন্যাস"
]

for w in words:

    print("\n", w)

    print(
        tok.encode(w).tokens
    )

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(
    tokenizer_file,
    test_file,
    vocab_size
):

    tokenizer = Tokenizer.from_file(
        tokenizer_file
    )

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                tokens = tokenizer.encode(
                    word
                ).tokens

                total_tokens += len(tokens)

                sent_tokens += len(tokens)

                total_chars += len(word)

                if len(tokens) > 1:
                    fragmented += 1

                if "[UNK]" in tokens:
                    oov += 1

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

wp_bn_results = []

for vocab_size in VOCABS:

    res = evaluate_tokenizer(
        f"wp_mix_bn_{vocab_size}.json",
        "test_mix_bn.txt",
        vocab_size
    )

    print(res)

    wp_bn_results.append(res)

print("\nFinished.")

In [ ]:
import csv

with open(
    "mix_bn_wp_results.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=wp_bn_results[0].keys()
    )

    writer.writeheader()

    writer.writerows(
        wp_bn_results
    )

print("Saved")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_mix_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_mix_iso_{vocab_size}.json"
    )

print("Finished")

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "wp_mix_iso_5000.json"
)

words = [
    "bāṃlā",
    "kalakātā",
    "sāhitya",
    "kabitā",
    "upanyāsa"
]

for w in words:

    print("\n", w)

    print(
        tok.encode(w).tokens
    )

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(
    tokenizer_file,
    test_file,
    vocab_size
):

    tokenizer = Tokenizer.from_file(
        tokenizer_file
    )

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                tokens = tokenizer.encode(
                    word
                ).tokens

                total_tokens += len(tokens)

                sent_tokens += len(tokens)

                total_chars += len(word)

                if len(tokens) > 1:
                    fragmented += 1

                if "[UNK]" in tokens:
                    oov += 1

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

wp_iso_results = []

for vocab_size in VOCABS:

    res = evaluate_tokenizer(
        f"wp_mix_iso_{vocab_size}.json",
        "test_mix_iso.txt",
        vocab_size
    )

    print(res)

    wp_iso_results.append(res)

print("\nFinished.")

In [ ]:
import csv

with open(
    "mix_iso_wp_results.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=wp_iso_results[0].keys()
    )

    writer.writeheader()
    writer.writerows(
        wp_iso_results
    )

print("Saved")

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining Bengali Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_mix_bn.txt",
        model_prefix=f"uni_mix_bn_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")

In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()

sp.load(
    "uni_mix_bn_5000.model"
)

words = [
    "বাংলাদেশ",
    "কলকাতা",
    "সাহিত্য",
    "কবিতা",
    "উপন্যাস"
]

for w in words:

    print("\n", w)

    print(
        sp.encode(
            w,
            out_type=str
        )
    )

In [ ]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(
    model_file,
    test_file,
    vocab_size
):

    sp = spm.SentencePieceProcessor()
    sp.load(model_file)

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                pieces = sp.encode(
                    word,
                    out_type=str
                )

                total_tokens += len(pieces)

                sent_tokens += len(pieces)

                total_chars += len(word)

                if len(pieces) > 1:
                    fragmented += 1

                if "<unk>" in pieces:
                    oov += 1

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
uni_bn_results = []

for vocab_size in VOCABS:

    res = evaluate_unigram(
        f"uni_mix_bn_{vocab_size}.model",
        "test_mix_bn.txt",
        vocab_size
    )

    print(res)

    uni_bn_results.append(res)

print("\nFinished.")

In [ ]:
import csv

with open(
    "mix_bn_uni_results.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=uni_bn_results[0].keys()
    )

    writer.writeheader()
    writer.writerows(
        uni_bn_results
    )

print("Saved")

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_mix_iso.txt",
        model_prefix=f"uni_mix_iso_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")

In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()

sp.load(
    "uni_mix_iso_5000.model"
)

words = [
    "bāṃlā",
    "kalakātā",
    "sāhitya",
    "kabitā",
    "upanyāsa"
]

for w in words:

    print("\n", w)

    print(
        sp.encode(
            w,
            out_type=str
        )
    )

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

uni_iso_results = []

for vocab_size in VOCABS:

    res = evaluate_unigram(
        f"uni_mix_iso_{vocab_size}.model",
        "test_mix_iso.txt",
        vocab_size
    )

    print(res)

    uni_iso_results.append(res)

print("\nFinished.")

In [ ]:
import csv

with open(
    "mix_iso_uni_results.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=uni_iso_results[0].keys()
    )

    writer.writeheader()

    writer.writerows(
        uni_iso_results
    )

print("Saved")

# Observations

## 1. Corpus Characteristics

The mixed corpus was created by combining equal portions of the news and literature datasets, resulting in a corpus containing 250,000 documents. The corpus contains 8.39 million words and a vocabulary of 671,371 unique word forms. The Type-Token Ratio (TTR) of 0.0800 lies between that of the news corpus (0.0621) and literature corpus (0.0933), indicating that the lexical diversity of the mixed corpus is intermediate between the two domains. Similarly, the average word length of 6.37 characters also falls between the values observed for news and literature data.

These statistics suggest that the mixed corpus successfully captures characteristics of both formal news writing and literary language, making it a suitable benchmark for evaluating tokenizer behavior across heterogeneous domains.

---

## 2. Effect of Transliteration

The Bengali corpus was transliterated into ISO format using the Aksharamukha transliteration framework. The transliterated corpus exhibited an expansion factor of 1.162, indicating that ISO representations require approximately 16.2% more characters than the original Bengali script. This increase is expected because several Bengali characters are represented by multiple Latin characters in ISO transliteration.

Despite the increase in character count, the vocabulary ratio between the ISO and Bengali versions remained relatively stable (0.962), suggesting that transliteration primarily changes orthographic representation rather than lexical content. The transliterated corpus therefore provides a useful alternative script representation while preserving the linguistic information present in the original corpus.

---

## 3. Impact of Vocabulary Size

Across all tokenization schemes, increasing the vocabulary size from 5,000 to 50,000 consistently improved tokenization efficiency. Fertility decreased steadily as vocabulary size increased, indicating that words were represented using fewer subword units. At the same time, Characters Per Token (CPT) increased, implying that learned tokens became longer and more semantically meaningful.

Word Fragmentation Rate (WFR) also decreased substantially with larger vocabularies, demonstrating that fewer words required decomposition into multiple subword segments. The reduction in token-count variance further suggests that tokenization became more stable and consistent across documents as the vocabulary size expanded.

These observations confirm the expected trade-off between vocabulary size and segmentation granularity: larger vocabularies reduce fragmentation and improve compression at the cost of increased model size.

---

## 4. Comparison of Tokenization Algorithms

### Bengali Script

For the Bengali corpus at a vocabulary size of 50,000, BPE achieved the lowest fertility (1.3186), the highest CPT (4.0728), and the lowest WFR (0.2420). WordPiece achieved slightly inferior performance, while Unigram produced the highest fertility and fragmentation among the three methods.

The ranking of tokenizer performance was:

BPE > WordPiece > Unigram

for all major efficiency metrics.

### ISO Script

A similar pattern was observed for the ISO-transliterated corpus. BPE again achieved the lowest fertility (1.3201) and WFR (0.2404), while WordPiece and Unigram followed in descending order of efficiency.

The consistency of these results across both scripts suggests that BPE is the most effective tokenizer among the evaluated approaches for Bengali-language text.

---

## 5. Influence of Transliteration on Tokenization

Comparing Bengali and ISO tokenization results reveals only minor differences in segmentation behavior. For example, at a vocabulary size of 50,000, Bengali BPE achieved a fertility of 1.3186, while ISO BPE achieved a fertility of 1.3201. Similar negligible differences were observed for WordPiece and Unigram.

Although transliteration increased character counts and CPT values, it did not significantly affect fertility or fragmentation rates. This suggests that tokenizer efficiency is influenced more strongly by lexical diversity and corpus characteristics than by the choice of script representation.

---

## 6. Cross-Domain Comparison

When compared with the previously evaluated news and literature corpora, the mixed corpus consistently produced intermediate results. For example, under BPE with a vocabulary size of 50,000:

| Domain | Fertility | CPT | WFR |
|----------|----------|----------|----------|
| News | 1.2745 | 4.3877 | 0.2102 |
| Mixed | 1.3186 | 4.0728 | 0.2420 |
| Literature | 1.3454 | 3.7880 | 0.2698 |

The same trend was observed across WordPiece and Unigram tokenizers. This behavior closely mirrors the lexical diversity measurements of the three corpora, where news exhibited the lowest TTR, literature the highest, and the mixed corpus occupied an intermediate position.

These findings indicate that lexical diversity has a direct influence on tokenization difficulty. As lexical diversity increases, fertility and fragmentation increase while compression efficiency decreases.

---

## 7. Overall Findings

The experimental results demonstrate that tokenizer performance is strongly influenced by both vocabulary size and corpus characteristics. Larger vocabularies consistently improve tokenization efficiency by reducing fragmentation and increasing token compactness. Among the evaluated tokenization methods, BPE consistently outperformed WordPiece and Unigram across all domains and scripts.

Furthermore, transliteration into ISO script had only a marginal effect on tokenization efficiency, suggesting that domain-specific lexical variation plays a substantially larger role than script representation. Finally, the mixed corpus behaved predictably between the news and literature domains, providing further evidence that lexical diversity is a key factor governing subword tokenization behavior.